# Notes importantes

## Rééchantillonnage du chemin

Si on ne rééchantillonne pas le tracé, l'altitude n'est lue qu'aux points d'origine. Entre deux points éloignés, la ligne passe alors en ligne droite à travers le relief, parfois sous une crête. La stratégie est d'ajouter des points intermédiaires, régulièrement espacés (par défaut tous les 20 m), pour que la ligne suive le terrain avec plus de détail.

Fichier : ./services/track.py

Le principe : on « déroule » le tracé sur une règle graduée en mètres (la distance parcourue depuis le départ), puis on pose un nouveau point à intervalles réguliers sur cette règle.

0. `d = numpy.diff(xz)`
   diff soustrait chaque ligne de la suivante : B - A, C - B, D - C. axis=0 veut dire « en descendant les lignes ». Avec 4 points, on obtient 3 écarts, un par segment.

1. `seg = numpy.hypot(d[:, 0], d[:, 1])`
   d[:, 0] est la colonne des écarts en x ([30, 0, 20]), d[:, 1] celle des écarts en z ([40, 30, 0]). hypot(dx, dz) calcule √(dx² + dz²) pour chaque segment, c’est Pythagore

2. Avec `cum = numpy.cumsum(seg)`
   Donne un tableau qui additionne les longueurs au fur et à mesure. Le dernier élément de ce tableau est la longueur totale du chemin.


In [ ]:
import numpy
from services.track import resample

# default track to numpy
xz = numpy.array([[0, 0], [30, 40], [30, 70], [50, 70]], dtype=float)
print(f"Tracé avec {len(xz)} points")
print(xz)

d = numpy.diff(xz, axis=0)
print("0. écart entre les points du segment")
print(d)

seg = numpy.hypot(d[:, 0], d[:, 1])
print("1. Longueur des segments")
print(seg)

cum = numpy.concatenate(([0.0], numpy.cumsum(seg)))
print("2. Tableau des longueurs au fur et à mesure du tracé")
print(cum)

Tracé avec 4 points
[[ 0.  0.]
 [30. 40.]
 [30. 70.]
 [50. 70.]]
0. écart entre les points du segment
[[30. 40.]
 [ 0. 30.]
 [20.  0.]]
1. Longueur des segments
[50. 30. 20.]
2. Tableau des longueurs au fur et à mesure du tracé
[  0.  50.  80. 100.]


3. `numpy.linspace(début, fin, nombre)` fabrique nombre valeurs régulièrement espacées, bornes comprises. Ansi on sait à quel longueur on veut un nouveau point.


In [ ]:
numpy.linspace(0, 10, 3)

array([ 0.,  5., 10.])

In [ ]:
numpy.linspace(0, 10, 6)

array([ 0.,  2.,  4.,  6.,  8., 10.])

In [ ]:
MAX_POINTS = 4000 # nombre de point max
STEP_M = 20.0 # espace voulu

total = cum[-1]
step = max(STEP_M, total / MAX_POINTS)
count = max(2, round(total / step) + 1)
targets = numpy.linspace(0, total, count)
print("3. On découpe en segment régulier")
print(targets)

3. On découpe en segment régulier
[  0.  20.  40.  60.  80. 100.]


In [ ]:
x = numpy.interp(targets, cum, xz[:, 0])
z = numpy.interp(targets, cum, xz[:, 1])
numpy.column_stack((x, z))

array([[ 0.,  0.],
       [12., 16.],
       [24., 32.],
       [30., 50.],
       [30., 70.],
       [50., 70.]])